In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install -U wandb
import wandb
from wandb.integration.keras import WandbMetricsLogger

# This will prompt you to paste your W&B API key
wandb.login()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.5/26.5 MB 59.6 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: wandb
    Found existing installation: wandb 0.26.1
    Uninstalling wandb-0.26.1:
      Successfully uninstalled wandb-0.26.1


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ayush1k (ayush1k-institute-of-engineering-and-technology-lucknow) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow import keras

In [4]:
import pathlib
import tensorflow as tf

# 1. Download the raw dataset directly to Kaggle
dataset_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
data_dir = tf.keras.utils.get_file('flower_photos', origin=dataset_url, untar=True)
data_dir = pathlib.Path(data_dir)

# 1. Fix the nested directory bug
if (data_dir / 'flower_photos').exists():
    data_dir = data_dir / 'flower_photos'
    
print("Using dataset path:", data_dir)

228813984/228813984 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Using dataset path: /root/.keras/datasets/flower_photos/flower_photos


In [5]:
sweep_config = {
    'method': 'grid',
    'metric': {'name': 'val_accuracy', 'goal': 'maximize'},
    'parameters': {
        'batch_size': {'values': [8]},
        'learning_rate': {'values': [0.0001]},
        'hidden_nodes': {'values': [128]},
        'img_size': {'values': [16]},
        'epochs': {'values': [10]},
        'experiment': {'values': ['dropout_only', 'batchnorm_only', 'full']}
    }
}

sweep_id = wandb.sweep(sweep_config, project="5-flowers-with-regularization-batchnorm")

Create sweep with ID: orba7n00
Sweep URL: https://wandb.ai/ayush1k-institute-of-engineering-and-technology-lucknow/5-flowers-with-regularization-batchnorm/sweeps/orba7n00


In [6]:
def train():
  with wandb.init() as run:
    config = wandb.config
    # Constants
    IMG_HEIGHT = config.img_size
    IMG_WIDTH = config.img_size
    IMG_CHANNELS = 3
    CLASS_NAMES = ["daisy", "dandelion", "roses", "sunflowers", "tulips"]

    def read_and_decode(filename, resize_dims):
        img_bytes = tf.io.read_file(filename)
        img = tf.image.decode_jpeg(img_bytes, channels=IMG_CHANNELS)
        img = tf.image.convert_image_dtype(img, tf.float32)
        img = tf.image.resize(img, resize_dims)
        return img

    def parse_csvline(csv_line):
        record_default = ["", ""]
        filename, label_string = tf.io.decode_csv(csv_line, record_default)
        img = read_and_decode(filename, [IMG_HEIGHT, IMG_WIDTH])
        label = tf.argmax(tf.math.equal(CLASS_NAMES, label_string))
        return img, label

    train_dataset = tf.keras.utils.image_dataset_from_directory(
      data_dir,
      validation_split=0.2,
      subset="training",
      seed=123,
      image_size=(IMG_HEIGHT, IMG_WIDTH),
      batch_size=config.batch_size)

    eval_dataset = tf.keras.utils.image_dataset_from_directory(
      data_dir,
      validation_split=0.2,
      subset="validation",
      seed=123,
      image_size=(IMG_HEIGHT, IMG_WIDTH),
      batch_size=config.batch_size)

    AUTOTUNE = tf.data.AUTOTUNE
    train_dataset = train_dataset.cache().prefetch(buffer_size=AUTOTUNE)
    eval_dataset = eval_dataset.cache().prefetch(buffer_size=AUTOTUNE)


# Build model
    model = keras.Sequential()
    model.add(keras.layers.Flatten(input_shape=(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS)))

    if config.experiment == 'batchnorm_only':
        model.add(keras.layers.Dense(config.hidden_nodes, use_bias=False))
        model.add(keras.layers.BatchNormalization())
        model.add(keras.layers.Activation("relu"))

    elif config.experiment == 'dropout_only':
        model.add(keras.layers.Dense(config.hidden_nodes, activation="relu"))
        model.add(keras.layers.Dropout(0.5))

    elif config.experiment == 'full':
        model.add(keras.layers.Dense(config.hidden_nodes,kernel_regularizer=keras.regularizers.l2(0.01),use_bias=False))
        model.add(keras.layers.BatchNormalization())
        model.add(keras.layers.Activation("relu"))
        model.add(keras.layers.Dropout(0.5))

    model.add(keras.layers.Dense(len(CLASS_NAMES), activation="softmax"))

    # Compile
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=config.learning_rate),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
        metrics=["accuracy"]
    )

        # Train
    callbacks = [WandbMetricsLogger(log_freq=5)]
    if config.experiment == 'full':
        callbacks.append(tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True))

    model.fit(
        train_dataset,
        validation_data=eval_dataset,
        epochs=config.epochs,
        callbacks=callbacks
    )

In [7]:
wandb.agent(sweep_id, function=train)

wandb: Agent Starting Run: lalnfi86 with config:
wandb: 	batch_size: 8
wandb: 	epochs: 10
wandb: 	experiment: dropout_only
wandb: 	hidden_nodes: 128
wandb: 	img_size: 16
wandb: 	learning_rate: 0.0001
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


Found 3670 files belonging to 5 classes.
Using 2936 files for training.


I0000 00:00:1784734188.744577     138 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Found 3670 files belonging to 5 classes.
Using 734 files for validation.


/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


...

Found 3670 files belonging to 5 classes.
Using 2936 files for training.
Found 3670 files belonging to 5 classes.
Using 734 files for validation.
Epoch 1/10


/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


367/367 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.2745 - loss: 4.0498 - val_accuracy: 0.3583 - val_loss: 3.5466
Epoch 2/10
367/367 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.3566 - loss: 3.6313 - val_accuracy: 0.4305 - val_loss: 3.3526
Epoch 3/10
367/367 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.4091 - loss: 3.3834 - val_accuracy: 0.4646 - val_loss: 3.1658
Epoch 4/10
367/367 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.4292 - loss: 3.2284 - val_accuracy: 0.4619 - val_loss: 3.0710
Epoch 5/10
367/367 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.4520 - loss: 3.0490 - val_accuracy: 0.4673 - val_loss: 2.9470
Epoch 6/10
367/367 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.4762 - loss: 2.9191 - val_accuracy: 0.4687 - val_loss: 2.8476
Epoch 7/10
367/367 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.4874 - loss: 2.7759 - val_accuracy: 0.4864 - val_loss: 2.7380
Epoch 8/10
367/367 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.5174 - loss: 2.6260 - val_accuracy: 0.4891 - val_

batch/accuracy,▃▃▃▃▁▄▄▁▅▅▅▅▅▅▅▅▅▆▇▇▆▇▆▆▆▆▆▇▇▇▇▇▇▇▇▇█▇▇▇
batch/batch_step,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇███
batch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
batch/loss,██▅▆▆▅▅▄▄▄▄▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/accuracy,▁▃▄▅▆▆▇▇██
epoch/epoch,▁▂▃▃▄▅▆▆▇█
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▆▅▄▄▃▃▂▁▁
epoch/val_accuracy,▁▅▆▆▆▇▇█▇█
epoch/val_loss,█▇▅▅▄▃▃▂▂▁
batch/accuracy,0.53381


wandb: Sweep Agent: Waiting for job.
wandb: Sweep Agent: Exiting.


In [8]:
print('done')

done


In [ ]:
https://wandb.ai/ayush1k-institute-of-engineering-and-technology-lucknow/5-flowers-with-regularization-batchnorm/panel/n32j16usd?nw=nwuserayush1k